# Setup environment

## Install required libraries

In [ ]:
%pip install -r requirements.txt

## Obtain an access token for the user

You must log in as a user to Azure CLI with appropriate permissions to create and run agents in Foundry.

In [1]:
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv('.env', override=True)

# Get a token for Microsoft Graph with the logged in user
credential = DefaultAzureCredential()
scopes = ["https://ai.azure.com/.default"]

user_token = credential.get_token(*scopes)

## (Optional) Setup up logging


In [2]:
# Setup logging in debug mode
import logging
import sys

def configure_logging(level="DEBUG"):
    """This function configures logging for code being run based on the specified level.
    Args:
        level (str): The logging level to use (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
    """
    try:
        # Convert the level string to uppercase so it matches what the logging library expects
        logging_level = getattr(logging, level.upper(), None)

        # Setup a logging format
        logging.basicConfig(
            level=logging_level,
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            handlers=[logging.StreamHandler(sys.stdout)]
        )
    except Exception as e:
        print(f"Failed to set up logging: {e}", file=sys.stderr)
        sys.exit(1) 

configure_logging("ERROR")

# Test a v2 agent

## Create the agent

In [5]:
import os
from dotenv import load_dotenv
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential
from azure.ai.projects.models import PromptAgentDefinition

load_dotenv(".env", override=True)

project_client = AIProjectClient(
    endpoint=os.environ["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential(),
)

with project_client.get_openai_client() as openai_client:
    agent = project_client.agents.create_version(
        agent_name="basic-v2-agent",
        definition=PromptAgentDefinition(
            model=os.environ["MODEL_DEPLOYMENT_NAME"],
            instructions="You are a helpful assistant that answers general questions",
        ),
    )
    print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

    conversation = openai_client.conversations.create(
        items=[{"type": "message", "role": "user", "content": "What is the size of France in square miles?"}],
    )
    print(f"Created conversation with initial user message (id: {conversation.id})")

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="",
    )
    print(f"Response output: {response.output_text}")

    openai_client.conversations.items.create(
        conversation_id=conversation.id,
        items=[{"type": "message", "role": "user", "content": "And what is the capital city?"}],
    )
    print(f"Added a second user message to the conversation")

    response = openai_client.responses.create(
        conversation=conversation.id,
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
        input="",
    )
    print(f"Response output: {response.output_text}")

    openai_client.conversations.delete(conversation_id=conversation.id)
    print("Conversation deleted")

project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
print("Agent deleted")

Agent created (id: basic-v2-agent:1, name: basic-v2-agent, version: 1)
Created conversation with initial user message (id: conv_41fa7ab031b44ede0086zqAWH0Fozse2FfNC8TWEnS3WOaWdfC)
Response output: France covers an area of approximately 248,573 square miles (643,801 square kilometers).
Added a second user message to the conversation
Response output: The capital city of France is Paris.
Conversation deleted
Agent deleted
